In [0]:
import os, sys, time
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from pyspark.sql import functions as F
from lh import common, silver

cases = [
    ("text", "  Star   Health ", "Star Health"), ("text", "N/A", None), ("text", "-", None),
    ("name", "  TARA CHOPRA ", "Tara Chopra"), ("code", " COMPLETED ", "completed"),
    ("city", "BLR", "Bengaluru"), ("city", "Bombay", "Mumbai"), ("city", "pune", "Pune"),
    ("phone", "+91-7937946724", "7937946724"), ("phone", "07937946724", "7937946724"),
    ("phone", "79379 46724", "7937946724"), ("phone", "12345", None),
    ("gender", "male", "M"), ("gender", "F", "F"), ("gender", "x", "O"),
    ("aadhaar", "082421729536", "082421729536"), ("aadhaar", "08242172953", None),
    ("integer", "250", "250"), ("integer", "abc", None),
    ("decimal", "₹1,23,456.50", "123456.50"), ("decimal", "700", "700.00"), ("decimal", "NULL", None),
    ("date", "2026-09-14", "2026-09-14"), ("date", "14/09/2026", "2026-09-14"),
    ("date", "14-Sep-2026", "2026-09-14"), ("date", "09/14/2026", "2026-09-14"),
    ("date", "14-09-2026", "2026-09-14"), ("date", "31/02/2026", None), ("date", "junk", None),
    ("date", "2026-10-07 14:41:49", "2026-10-07"),
    ("timestamp", "2026-10-07 14:41:49", "2026-10-07 14:41:49"),
    ("timestamp", "2026-10-07T14:41:49", "2026-10-07 14:41:49"),
    ("timestamp", "07-10-2026 14:41:49", "2026-10-07 14:41:49"),
]
df = spark.createDataFrame([(i, c[1]) for i, c in enumerate(cases)], "i int, raw string")
kinds = sorted({c[0] for c in cases})
out = df.select("i", *[getattr(silver, k)(F.col("raw")).cast("string").alias(k) for k in kinds]).orderBy("i").collect()
fails = [(c, out[i][c[0]]) for i, c in enumerate(cases) if out[i][c[0]] != c[2]]
print(len(cases), "cases,", len(fails), "failed", fails)
print("PASS" if not fails else "FAIL")

In [0]:
cfg = common.load_config("dev")
start = time.time()
print(f"{'table':18} {'rows':>6} {'bad':>4} {'deleted':>7}  unreadable values by column")
total_bad = 0
for table, spec in silver.load_silver().items():
    out = silver.apply(spark.table(f"{cfg['catalog']}.bronze.{table}"), spec)
    r = out.agg(F.count("*").alias("n"),
                F.sum(F.when(F.size("_bad") > 0, 1).otherwise(0)).alias("bad"),
                F.sum(F.col("is_deleted").cast("int")).alias("deleted")).first()
    cols = {x["c"]: x["count"] for x in out.select(F.explode("_bad").alias("c")).groupBy("c").count().collect()}
    total_bad += r["bad"]
    print(f"{table:18} {r['n']:>6} {r['bad']:>4} {r['deleted']:>7}  {cols or ''}")
print(f"done in {time.time() - start:.0f}s", "PASS" if total_bad == 0 else "CHECK")